In [1]:
import csv
import os
import pandas as pd

def mapping(DIZIONARIO_VECCHIO_TARGET, DIZIONARI_NUOVI_TARGETS):

    # ==============================================================================
    # CONFIGURAZIONE DEL CONFRONTO CORRELAZIONALE (1-a-N) E REPORTING
    # ==============================================================================
    file_master_nuovo = "../openbis_reference_vocabularies/external_dictionaries_merged.csv"
    file_master_vecchio = "../before_mapping/metadata-model-rockmechanics_V3_spaces_vocabularies.csv"


    # 3. Gestione DINAMICA della sottocartella di output con il nome del dizionario vecchio
    output_dir = DIZIONARIO_VECCHIO_TARGET
    os.makedirs(output_dir, exist_ok=True) # Crea la cartella (es. "MSL_ROCK/") se non esiste

    file_report_mappati = os.path.join(output_dir, "report_1toN_mappati.csv")
    file_report_esclusi = os.path.join(output_dir, "report_1toN_esclusi.csv")
    # ==============================================================================

    def estrai_blocco_dizionario(file_path, target_code):
        """Estrae un dizionario specifico mappando dinamicamente le colonne rilevate"""
        records = []
        current_vocab = None
        headers = None
        
        if not os.path.exists(file_path):
            return pd.DataFrame()
            
        with open(file_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
            reader = csv.reader(f, delimiter=';')
            for row in reader:
                if not row or all(cell.strip() == "" for cell in row):
                    continue
                    
                if "VOCABULARY_TYPE" in row[0]:
                    current_vocab = None
                    headers = None
                    continue
                
                if current_vocab is None and len(row) >= 2 and row[1].strip() == target_code:
                    current_vocab = target_code
                    continue
                
                if current_vocab == target_code:
                    if "Label" in row and "Description" in row:
                        headers = [c.strip() for c in row]
                        continue
                    
                    if headers is not None:
                        record = {}
                        for idx, cell in enumerate(row):
                            if idx < len(headers):
                                record[headers[idx]] = cell.strip()
                        
                        if "Label" in record and record["Label"]:
                            # Tracciabilità del modulo di origine
                            record["Vocab_Source"] = target_code
                            records.append(record)
                            
        return pd.DataFrame(records)

    print(f"Avvio Analisi Comparativa 1-a-N per: '{DIZIONARIO_VECCHIO_TARGET}'")
    print(f"Destinazioni di ricerca controllata: {DIZIONARI_NUOVI_TARGETS}")
    print("=" * 80)

    # 1. Estrazione del vecchio dizionario singolo
    df_old = estrai_blocco_dizionario(file_master_vecchio, DIZIONARIO_VECCHIO_TARGET)

    # 2. Estrazione e fusione degli N dizionari nuovi in un unico paniere
    lista_df_nuovi = []
    for nuovo_codice in DIZIONARI_NUOVI_TARGETS:
        df_singolo_nuovo = estrai_blocco_dizionario(file_master_nuovo, nuovo_codice)
        if not df_singolo_nuovo.empty:
            lista_df_nuovi.append(df_singolo_nuovo)

    df_new_combined = pd.concat(lista_df_nuovi, ignore_index=True) if lista_df_nuovi else pd.DataFrame()

    if df_old.empty:
        print(f"[ERRORE] Impossibile trovare il vecchio dizionario '{DIZIONARIO_VECCHIO_TARGET}' nel file master vecchio.")
    elif df_new_combined.empty:
        print(f"[ERRORE] Nessuno dei dizionari nuovi indicati {DIZIONARI_NUOVI_TARGETS} è stato trovato nel file unificato.")
    else:
        print(f"[INFO] Record nel VECCHIO dizionario: {len(df_old)}")
        print(f"[INFO] Record totali combinati nei {len(lista_df_nuovi)} NUOVI dizionari: {len(df_new_combined)}")
        print("-" * 80)
        
        # Normalizzazione per il confronto case-insensitive
        df_old['label_lower'] = df_old['Label'].str.lower()
        df_new_combined['label_lower'] = df_new_combined['Label'].str.lower()
        
        # 3. CONFRONTO INCROCIATO 1-A-N
        df_match = pd.merge(df_old, df_new_combined, on='label_lower', suffixes=('_vecchio', '_nuovo'))
        esclusi = df_old[~df_old['label_lower'].isin(df_new_combined['label_lower'])]
        
        # 4. REPORT A SCHERMO
        print(f"DIAGNOSTICA DI TRANSIZIONE PER '{DIZIONARIO_VECCHIO_TARGET}':")
        print(f" ✅ Termini riallocati con successo nelle nuove ontologie: {len(df_match)}")
        print(f" ❌ Termini rimasti fuori da questa selezione di dizionari:  {len(esclusi)}")
        print("-" * 80)
        
        if not df_match.empty:
            print("\nDistribuzione dei tuoi vecchi termini nelle nuove destinazioni:")
            conteggio_destinazioni = df_match['Vocab_Source_nuovo'].value_counts()
            for vocab, count in conteggio_destinazioni.items():
                print(f"   • Finiti in {vocab}: {count} termini")
                
            print("\nEsempio di riallocazione dei codici e degli URI:")
            for _, row in df_match.head(4).iterrows():
                print(f"   • [{row['Label_vecchio']}] ")
                print(f"     Da: {row['Vocab_Source_vecchio']} ({row['Code_vecchio']})")
                print(f"     A : {row['Vocab_Source_nuovo']} ({row['Code_nuovo']}) -> URI: {row['Description_nuovo']}")
                
        if not esclusi.empty:
            print(f"\nAnteprima dei termini rimasti fuori (primi 5):")
            for _, row in esclusi.head(5).iterrows():
                print(f"   • {row['Label']} (Vecchio Codice: {row['Code']})")

        # ==============================================================================
        # 5. SEZIONE DI SALVATAGGIO DEI REPORT NELLA SOTTOCARTELLA DEDICATA
        # ==============================================================================
        print("\n" + "=" * 80)
        print(f"5. ESPORTAZIONE DEI FILE DI REPORT NELLA CARTELLA '{output_dir}/'")
        print("=" * 80)
        
        # A. Esportazione dei promossi
        if not df_match.empty:
            colonne_output_match = [
                'Label_vecchio', 'Vocab_Source_vecchio', 'Code_vecchio',
                'Vocab_Source_nuovo', 'Code_nuovo', 'Description_nuovo'
            ]
            colonne_esistenti = [c for c in colonne_output_match if c in df_match.columns]
            
            df_match_report = df_match[colonne_esistenti].rename(columns={
                'Label_vecchio': 'Label',
                'Vocab_Source_vecchio': 'Dizionario_Vecchio',
                'Code_vecchio': 'Codice_Vecchio',
                'Vocab_Source_nuovo': 'Dizionario_Nuovo',
                'Code_nuovo': 'Codice_Nuovo',
                'Description_nuovo': 'Nuovo_URI_Pubblico'
            })
            
            df_match_report.to_csv(file_report_mappati, sep=';', index=False, encoding='utf-8-sig')
            print(f" 💾 Salto a buon fine: '{file_report_mappati}'")
            
        # B. Esportazione degli orfani esclusi
        if not esclusi.empty:
            df_esclusi_report = esclusi[['Code', 'Description', 'Label', 'Vocab_Source']].rename(columns={
                'Vocab_Source': 'Dizionario_Sorgente',
                'Description': 'Vecchio_Contenuto_Descrizione'
            })
            df_esclusi_report.to_csv(file_report_esclusi, sep=';', index=False, encoding='utf-8-sig')
            print(f" 💾 Salto a buon fine: '{file_report_esclusi}'")

    print("=" * 80)

In [2]:
mapping("MSL_ROCK" , ["MSL_ROCK"])

Avvio Analisi Comparativa 1-a-N per: 'MSL_ROCK'
Destinazioni di ricerca controllata: ['MSL_ROCK']
[INFO] Record nel VECCHIO dizionario: 204
[INFO] Record totali combinati nei 1 NUOVI dizionari: 180
--------------------------------------------------------------------------------
DIAGNOSTICA DI TRANSIZIONE PER 'MSL_ROCK':
 ✅ Termini riallocati con successo nelle nuove ontologie: 182
 ❌ Termini rimasti fuori da questa selezione di dizionari:  24
--------------------------------------------------------------------------------

Distribuzione dei tuoi vecchi termini nelle nuove destinazioni:
   • Finiti in MSL_ROCK: 182 termini

Esempio di riallocazione dei codici e degli URI:
   • [breccia - fault related] 
     Da: MSL_ROCK (MSL_ROCK1)
     A : MSL_ROCK (MSL_ROCK1) -> URI: https://epos-msl.uu.nl/voc/materials/1.3/fault_rock-breccia_-_fault_related
   • [cataclasite] 
     Da: MSL_ROCK (MSL_ROCK2)
     A : MSL_ROCK (MSL_ROCK2) -> URI: https://epos-msl.uu.nl/voc/materials/1.3/fault_rock-cata

In [3]:
mapping("UNITS" , ["QUDT_UNIT"])

Avvio Analisi Comparativa 1-a-N per: 'UNITS'
Destinazioni di ricerca controllata: ['QUDT_UNIT']
[INFO] Record nel VECCHIO dizionario: 111
[INFO] Record totali combinati nei 1 NUOVI dizionari: 2894
--------------------------------------------------------------------------------
DIAGNOSTICA DI TRANSIZIONE PER 'UNITS':
 ✅ Termini riallocati con successo nelle nuove ontologie: 88
 ❌ Termini rimasti fuori da questa selezione di dizionari:  23
--------------------------------------------------------------------------------

Distribuzione dei tuoi vecchi termini nelle nuove destinazioni:
   • Finiti in QUDT_UNIT: 88 termini

Esempio di riallocazione dei codici e degli URI:
   • [second] 
     Da: UNITS (UNI1)
     A : QUDT_UNIT (QUDT_UNIT2475) -> URI: http://qudt.org/vocab/unit/SEC
   • [metre] 
     Da: UNITS (UNI2)
     A : QUDT_UNIT (QUDT_UNIT1342) -> URI: http://qudt.org/vocab/unit/M
   • [kilogram] 
     Da: UNITS (UNI3)
     A : QUDT_UNIT (QUDT_UNIT1029) -> URI: http://qudt.org/vocab/un

In [4]:
mapping("PHYSICALQUANTITIES" , ["QUDT_QUANTITYKIND"])

Avvio Analisi Comparativa 1-a-N per: 'PHYSICALQUANTITIES'
Destinazioni di ricerca controllata: ['QUDT_QUANTITYKIND']
[INFO] Record nel VECCHIO dizionario: 103
[INFO] Record totali combinati nei 1 NUOVI dizionari: 1007
--------------------------------------------------------------------------------
DIAGNOSTICA DI TRANSIZIONE PER 'PHYSICALQUANTITIES':
 ✅ Termini riallocati con successo nelle nuove ontologie: 82
 ❌ Termini rimasti fuori da questa selezione di dizionari:  25
--------------------------------------------------------------------------------

Distribuzione dei tuoi vecchi termini nelle nuove destinazioni:
   • Finiti in QUDT_QUANTITYKIND: 82 termini

Esempio di riallocazione dei codici e degli URI:
   • [Length] 
     Da: PHYSICALQUANTITIES (PHQ1)
     A : QUDT_QUANTITYKIND (QUDT_QUANTITYKIND433) -> URI: http://qudt.org/vocab/quantitykind/Length
   • [Mass] 
     Da: PHYSICALQUANTITIES (PHQ2)
     A : QUDT_QUANTITYKIND (QUDT_QUANTITYKIND503) -> URI: http://qudt.org/vocab/quant

In [5]:
mapping("ROCKS" , ["BGS_ROCKNAME"])

Avvio Analisi Comparativa 1-a-N per: 'ROCKS'
Destinazioni di ricerca controllata: ['BGS_ROCKNAME']
[INFO] Record nel VECCHIO dizionario: 182
[INFO] Record totali combinati nei 1 NUOVI dizionari: 3113
--------------------------------------------------------------------------------
DIAGNOSTICA DI TRANSIZIONE PER 'ROCKS':
 ✅ Termini riallocati con successo nelle nuove ontologie: 84
 ❌ Termini rimasti fuori da questa selezione di dizionari:  98
--------------------------------------------------------------------------------

Distribuzione dei tuoi vecchi termini nelle nuove destinazioni:
   • Finiti in BGS_ROCKNAME: 84 termini

Esempio di riallocazione dei codici e degli URI:
   • [Andesite] 
     Da: ROCKS (ROC2)
     A : BGS_ROCKNAME (BGS_ROCKNAME25) -> URI: http://data.bgs.ac.uk/id/EarthMaterialClass/RockName/AND
   • [Anorthosite] 
     Da: ROCKS (ROC4)
     A : BGS_ROCKNAME (BGS_ROCKNAME56) -> URI: http://data.bgs.ac.uk/id/EarthMaterialClass/RockName/ANO
   • [Basalt] 
     Da: ROCKS 

In [6]:
mapping("ARTIFICIALMATERIALS" , ["SWEET_MATR_CORE", "SWEET_MATR_IND"])

Avvio Analisi Comparativa 1-a-N per: 'ARTIFICIALMATERIALS'
Destinazioni di ricerca controllata: ['SWEET_MATR_CORE', 'SWEET_MATR_IND']
[INFO] Record nel VECCHIO dizionario: 8
[INFO] Record totali combinati nei 2 NUOVI dizionari: 37
--------------------------------------------------------------------------------
DIAGNOSTICA DI TRANSIZIONE PER 'ARTIFICIALMATERIALS':
 ✅ Termini riallocati con successo nelle nuove ontologie: 1
 ❌ Termini rimasti fuori da questa selezione di dizionari:  7
--------------------------------------------------------------------------------

Distribuzione dei tuoi vecchi termini nelle nuove destinazioni:
   • Finiti in SWEET_MATR_IND: 1 termini

Esempio di riallocazione dei codici e degli URI:
   • [Alloy] 
     Da: ARTIFICIALMATERIALS (ARTMAT2)
     A : SWEET_MATR_IND (SWEET_MATR_IND1) -> URI: http://sweetontology.net/matrIndustrial/Alloy

Anteprima dei termini rimasti fuori (primi 5):
   • Adhesive (Vecchio Codice: ARTMAT1)
   • Ceramic (Vecchio Codice: ARTMAT3)

In [7]:
mapping("FLUIDS" , ["SWEET_STATE_PHYS"])

Avvio Analisi Comparativa 1-a-N per: 'FLUIDS'
Destinazioni di ricerca controllata: ['SWEET_STATE_PHYS']
[INFO] Record nel VECCHIO dizionario: 3
[INFO] Record totali combinati nei 1 NUOVI dizionari: 83
--------------------------------------------------------------------------------
DIAGNOSTICA DI TRANSIZIONE PER 'FLUIDS':
 ✅ Termini riallocati con successo nelle nuove ontologie: 2
 ❌ Termini rimasti fuori da questa selezione di dizionari:  1
--------------------------------------------------------------------------------

Distribuzione dei tuoi vecchi termini nelle nuove destinazioni:
   • Finiti in SWEET_STATE_PHYS: 2 termini

Esempio di riallocazione dei codici e degli URI:
   • [Liquid] 
     Da: FLUIDS (FLUID1)
     A : SWEET_STATE_PHYS (SWEET_STATE_PHYS46) -> URI: http://sweetontology.net/statePhysical/Liquid
   • [Gas] 
     Da: FLUIDS (FLUID2)
     A : SWEET_STATE_PHYS (SWEET_STATE_PHYS24) -> URI: http://sweetontology.net/statePhysical/Gas

Anteprima dei termini rimasti fuori (pr